# Wheat Disease Model Training (Google Colab)

## Steps:
1. Upload your `WheatDisease` folder to Google Drive
2. Run all cells below
3. Download the trained model back to your local project

**Expected time:** ~30-40 minutes on T4 GPU (vs 4.5 hours on CPU)

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Check GPU

In [ ]:
import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")
!nvidia-smi

## Step 3: Set Data Path

**IMPORTANT:** Upload your `WheatDisease` folder to Google Drive first.

Structure should be:
```
Google Drive/WheatDisease/WheatDisease/wheat/
    healthy/
    wheat_aphid/
    wheat_black_rust/
    ...
```

In [ ]:
import os

# Change this path if your folder is in a different location on Google Drive
DATA_DIR = "/content/drive/MyDrive/WheatDisease/WheatDisease"

# Verify the data exists
wheat_dir = os.path.join(DATA_DIR, "wheat")
if os.path.exists(wheat_dir):
    classes = sorted(os.listdir(wheat_dir))
    print(f"Found {len(classes)} classes:")
    for c in classes:
        count = len(os.listdir(os.path.join(wheat_dir, c)))
        print(f"  {c}: {count} images")
else:
    print(f"ERROR: {wheat_dir} not found!")
    print("Upload WheatDisease folder to Google Drive first.")

## Step 4: Pool Data (merge crop subfolder into flat class structure)

In [ ]:
import shutil
from pathlib import Path

data_dir = Path(DATA_DIR)
pooled_dir = Path("/content/wheat_disease_pool")

# Pool: copy from wheat/<class>/*.jpg to pool/<class>/*.jpg
if pooled_dir.exists():
    shutil.rmtree(pooled_dir)
pooled_dir.mkdir(parents=True)

found = 0
for crop_dir in data_dir.iterdir():
    if not crop_dir.is_dir():
        continue
    for disease_dir in crop_dir.iterdir():
        if not disease_dir.is_dir():
            continue
        dst = pooled_dir / disease_dir.name
        dst.mkdir(parents=True, exist_ok=True)
        for img in disease_dir.glob("*.*"):
            if img.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp"):
                shutil.copy(img, dst / f"{crop_dir.name}_{img.name}")
                found += 1

print(f"Pooled {found} images into {pooled_dir}")
for d in sorted(pooled_dir.iterdir()):
    print(f"  {d.name}: {len(list(d.glob('*.*')))} images")

## Step 5: Train the Model

In [ ]:
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# === Config ===
IMG_SIZE = (224, 224)
BATCH_SIZE = 32  # Can use larger batch on GPU
EPOCHS = 25
CROP = "wheat"

# === Data Generators ===
datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.25,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3],
    fill_mode="nearest",
    validation_split=0.2,
)

train_gen = datagen.flow_from_directory(
    pooled_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    shuffle=True,
)

val_gen = datagen.flow_from_directory(
    pooled_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    shuffle=False,
)

num_classes = train_gen.num_classes
class_indices = train_gen.class_indices
labels_by_index = {v: k for k, v in class_indices.items()}

print(f"\nClasses ({num_classes}): {list(class_indices.keys())}")
print(f"Training samples: {train_gen.samples}")
print(f"Validation samples: {val_gen.samples}")

# === Class Weights ===
total = sum(train_gen.classes.tolist().count(i) for i in range(num_classes))
class_weights = {}
for i in range(num_classes):
    count = train_gen.classes.tolist().count(i)
    class_weights[i] = total / (num_classes * count) if count > 0 else 1.0
print(f"\nClass weights computed (min: {min(class_weights.values()):.2f}, max: {max(class_weights.values()):.2f})")

In [ ]:
# === Build Model ===
base_model = EfficientNetB0(
    input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet"
)
base_model.trainable = False

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)
model = models.Model(inputs, outputs)

model.summary()
print(f"\nTotal params: {model.count_params():,}")

In [ ]:
# === Callbacks ===
MODEL_PATH = "/content/disease_model_wheat.keras"

best_checkpoint = ModelCheckpoint(
    MODEL_PATH, monitor="val_accuracy", save_best_only=True, verbose=1
)
early_stop = EarlyStopping(
    monitor="val_accuracy", patience=5, restore_best_weights=True, verbose=1
)
reduce_lr = ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
)

# === Phase 1: Train top layers only ===
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"],
)

PHASE1_EPOCHS = 8
print(f"\n{'='*60}")
print(f"  PHASE 1: Training top layers for {PHASE1_EPOCHS} epochs")
print(f"{'='*60}")

history_phase1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=PHASE1_EPOCHS,
    class_weight=class_weights,
    callbacks=[best_checkpoint, reduce_lr],
)

In [ ]:
# === Phase 2: Fine-tune top 20 layers (BatchNorm frozen) ===
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False
# Keep BatchNorm frozen to prevent statistics shift
for layer in base_model.layers[-20:]:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

trainable_count = sum(1 for l in model.layers if l.trainable)
print(f"Trainable layers: {trainable_count}")

# Warmup LR schedule
class WarmupSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, target_lr=5e-5, warmup_epochs=2, steps_per_epoch=100):
        super().__init__()
        self.target_lr = target_lr
        self.warmup_steps = warmup_epochs * steps_per_epoch
        self.total_warmup = float(self.warmup_steps)

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        return self.target_lr * tf.minimum(step / self.total_warmup, 1.0)

    def get_config(self):
        return {"target_lr": self.target_lr, "warmup_steps": self.warmup_steps}

warmup_lr = WarmupSchedule(
    target_lr=5e-5,
    warmup_epochs=2,
    steps_per_epoch=len(train_gen),
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=warmup_lr),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"],
)

PHASE2_EPOCHS = EPOCHS - PHASE1_EPOCHS
print(f"\n{'='*60}")
print(f"  PHASE 2: Fine-tuning for {PHASE2_EPOCHS} epochs (warmup LR)")
print(f"{'='*60}")

history_phase2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=PHASE2_EPOCHS,
    class_weight=class_weights,
    callbacks=[best_checkpoint, early_stop],
)

## Step 6: View Results

In [ ]:
import matplotlib.pyplot as plt

# Merge histories
full_history = {}
for key in history_phase1.history:
    full_history[key] = history_phase1.history[key] + history_phase2.history.get(key, [])

epochs_range = range(1, len(full_history["accuracy"]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Wheat Disease Model - Training Log", fontsize=14, fontweight="bold")

# Accuracy
ax1.plot(epochs_range, full_history["accuracy"], "b-o", markersize=4, label="Train")
ax1.plot(epochs_range, full_history["val_accuracy"], "r-s", markersize=4, label="Validation")
best_val = max(full_history["val_accuracy"])
best_ep = full_history["val_accuracy"].index(best_val) + 1
ax1.axhline(y=best_val, color="green", linestyle="--", alpha=0.5)
ax1.scatter([best_ep], [best_val], color="green", s=100, zorder=5, marker="*")
ax1.set_title(f"Accuracy (Best Val: {best_val:.4f} @ Epoch {best_ep})")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Loss
ax2.plot(epochs_range, full_history["loss"], "b-o", markersize=4, label="Train")
ax2.plot(epochs_range, full_history["val_loss"], "r-s", markersize=4, label="Validation")
ax2.set_title("Loss")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/content/wheat_disease_training_chart.png", dpi=150)
plt.show()

# Print summary
print(f"\n{'='*60}")
print(f"  TRAINING COMPLETE - Wheat Disease Model")
print(f"{'='*60}")
print(f"  Best Validation Accuracy: {best_val:.4f} (Epoch {best_ep})")
print(f"  Final Train Accuracy:     {full_history['accuracy'][-1]:.4f}")
print(f"  Final Val Accuracy:       {full_history['val_accuracy'][-1]:.4f}")
print(f"  Overfit Gap:              {full_history['accuracy'][-1] - full_history['val_accuracy'][-1]:.4f}")
print(f"{'='*60}")

## Step 7: Save Labels + Copy to Google Drive

In [ ]:
# Save labels
LABELS_PATH = "/content/disease_labels_wheat.json"
with open(LABELS_PATH, "w") as f:
    json.dump(labels_by_index, f, indent=2)

# Save training history
HISTORY_PATH = "/content/wheat_disease_training_history.json"
with open(HISTORY_PATH, "w") as f:
    json.dump(full_history, f, indent=2)

# Copy to Google Drive for safekeeping
drive_output = "/content/drive/MyDrive/WheatDisease/trained_model"
os.makedirs(drive_output, exist_ok=True)

shutil.copy(MODEL_PATH, os.path.join(drive_output, "disease_model_wheat.keras"))
shutil.copy(LABELS_PATH, os.path.join(drive_output, "disease_labels_wheat.json"))
shutil.copy(HISTORY_PATH, os.path.join(drive_output, "wheat_disease_training_history.json"))
shutil.copy("/content/wheat_disease_training_chart.png", os.path.join(drive_output, "wheat_disease_training_chart.png"))

print(f"\nAll files saved to Google Drive: {drive_output}")
print(f"  - disease_model_wheat.keras")
print(f"  - disease_labels_wheat.json")
print(f"  - wheat_disease_training_history.json")
print(f"  - wheat_disease_training_chart.png")
print(f"\nLabels: {labels_by_index}")

## Step 8: Download to Local Machine

Run this cell to download directly, OR copy from Google Drive manually.

In [ ]:
from google.colab import files

# Download model and labels
files.download(MODEL_PATH)
files.download(LABELS_PATH)
files.download(HISTORY_PATH)
files.download("/content/wheat_disease_training_chart.png")

print("\n" + "="*60)
print("  NEXT STEPS:")
print("="*60)
print("  1. Copy downloaded files to your local project:")
print("     disease_model_wheat.keras  -> D:\\AI_Projects\\AI\\models\\")
print("     disease_labels_wheat.json  -> D:\\AI_Projects\\AI\\models\\")
print("  2. Start your server:")
print("     uvicorn main:app --host 0.0.0.0 --port 8001")
print("  3. Test:")
print("     POST /predict/disease with crop=wheat and a wheat leaf image")
print("="*60)